# HiSparse 入门 (5/8)：656B 行格式 —— A5 上的 fp8 C8 长什么样

> **一句话结论**：一行固定 656B，分三段：512B NoPE 量化值 + 128B BF16 RoPE + 16B FP32 scales；A5 用硬件原生 E4M3FN fp8 存 NoPE。

## 1. 三段拆解

| 段 | 字节 | 类型 | 装什么 |
| --- | --- | --- | --- |
| NoPE（无位置信息内容） | 512B | E4M3FN fp8 | 量化后的 KV 主值 |
| RoPE（位置信息） | 128B | BF16 | 64 维 × 2B，位置编码分量 |
| scales | 16B | FP32 ×4 | 4 个块的量化缩放系数 |

合计：512 + 128 + 16 = **656B**。

## 2. 字节校验（纯计算）


In [ ]:
nope_bytes = 512      # E4M3FN fp8, 1 byte/element
rope_bytes = 128     # BF16, 2 bytes/element
scale_bytes = 4 * 4  # 4 x FP32

row_bytes = nope_bytes + rope_bytes + scale_bytes
print("row bytes:", row_bytes)
assert row_bytes == 656


## 3. A5 的硬件前提

- A5（Ascend 950）支持**原生 E4M3FN fp8**，块量化可由既有链/硬件完成；
- Main 与 Indexer 的 dtype 可以独立选择；
- 注意力在池上按物理行号原地直读这 656B，不做二次重组。

## 4. 一个比喻

一行就是一个标准信封：

- 主内容（NoPE）占最大格子；
- 位置条（RoPE）单独一格，避免被量化破坏；
- 每块的换算系数（scales）贴在信封背面。

## 5. 小结

- 656B = 512B fp8 NoPE + 128B BF16 RoPE + 16B FP32 scales；
- A5 依赖硬件原生 fp8；这一点到 A3 会成为关键差异；
- 下一篇把这些部件串成一步完整的前向流程。

**下一篇**：`06-forward-flow-a5.ipynb`
